# 02 · SWH WPP 573, 2021–2025

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import statsmodels.api as sm
from scipy import optimize, stats
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import swh_core as core

RAW       = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
TABLES    = ROOT / "output" / "tables"
FIGURES   = ROOT / "output" / "figures"
for d in (PROCESSED, TABLES, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

WPP        = "573"
TAHUN_AWAL, TAHUN_AKHIR = core.tahun_lengkap_terakhir()
T          = 365.25
JEDA       = 2
P_AMBANG   = 90
LEBAR_ZONA = 4.0

plt.rcParams.update({"figure.dpi": 120, "font.size": 9})
pd.set_option("display.width", 130, "display.max_columns", 40)

print(f"WPP {WPP} | jendela {TAHUN_AWAL}-{TAHUN_AKHIR} | ambang p{P_AMBANG} | jeda dekluster {JEDA} hari")

## load, classify cells

In [ ]:
berkas = sorted(f for f in RAW.glob(f"era5_swh_wpp{WPP}_*.nc")
                if (th := core.tahun_dari_nama(f)) and TAHUN_AWAL <= th <= TAHUN_AKHIR)
print(f"{len(berkas)} berkas:", ", ".join(f.name for f in berkas))

da = xr.concat([xr.open_dataset(f)["swh"] for f in berkas], dim="valid_time")
da = da.sortby("valid_time").rename({"valid_time": "waktu"}).load()

waktu = pd.to_datetime(da.waktu.values)
n_tahun = waktu.year.nunique()
print(f"\n{da.sizes['waktu']:,} hari, {waktu.min().date()} s/d {waktu.max().date()}, {n_tahun} tahun")
print(f"grid: {da.sizes['latitude']} lintang x {da.sizes['longitude']} bujur, resolusi 0,5 derajat")

In [ ]:
laut = da.notnull().all("waktu").values
darat = ~laut
nlat, nlon = laut.shape

pesisir = np.zeros_like(laut)
for i in range(nlat):
    for j in range(nlon):
        if laut[i, j] and (darat[max(0, i-1):i+2, max(0, j-1):j+2].any()
                           or i in (0, nlat-1) or j in (0, nlon-1)):
            pesisir[i, j] = True
lepas = laut & ~pesisir

print(f"{laut.size} sel total -> {laut.sum()} laut, {darat.sum()} darat")
print(f"   {pesisir.sum()} pesisir (domain tier <5 GT), {lepas.sum()} lepas pantai")

## heterogeneity

In [ ]:
nilai = da.values
idx_pes = np.argwhere(pesisir)
mat_pes = np.stack([nilai[:, i, j] for i, j in idx_pes], axis=1)
lon_pes = da.longitude.values[idx_pes[:, 1]]
lat_pes = da.latitude.values[idx_pes[:, 0]]

hari_2m = (mat_pes > 2.0).sum(axis=0) / n_tahun
print("Hari SWH > 2,0 m per tahun, antar-sel pesisir:")
print(pd.Series(hari_2m).describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_string())

In [ ]:
tepi = np.arange(104, 129, LEBAR_ZONA)
zona = np.digitize(lon_pes, tepi)

ringkas_zona = pd.DataFrame({"zona": zona, "lon": lon_pes, "hari_2m": hari_2m})
tab_het = (ringkas_zona.groupby("zona")
           .agg(bujur_min=("lon", "min"), bujur_maks=("lon", "max"),
                n_sel=("lon", "size"), hari_2m_rata=("hari_2m", "mean"),
                hari_2m_sd=("hari_2m", "std"))
           .round(1))
tab_het

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1.6, 1]})
sc = ax[0].scatter(lon_pes, lat_pes, c=hari_2m, s=16, cmap="YlOrRd", vmin=0)
fig.colorbar(sc, ax=ax[0], label="hari SWH > 2 m / tahun")
ax[0].set_xlabel("Bujur"); ax[0].set_ylabel("Lintang")
ax[0].set_title("Sel pesisir WPP 573", fontsize=10)
for x in tepi[1:-1]:
    ax[0].axvline(x, color="k", lw=0.4, alpha=0.3)

ax[1].boxplot([ringkas_zona.loc[ringkas_zona.zona == z, "hari_2m"] for z in sorted(set(zona))],
              tick_labels=[f"Z{z}" for z in sorted(set(zona))])
ax[1].set_ylabel("hari > 2 m / tahun")
ax[1].set_title("Sebaran per zona", fontsize=10)
fig.tight_layout()
fig.savefig(FIGURES / f"fig_heterogenitas_wpp{WPP}.png", dpi=160, bbox_inches="tight")
plt.show()

## zone index + relative threshold

In [ ]:
zona_unik = sorted(z for z in set(zona) if (zona == z).sum() >= 3)

indeks, ambang = {}, {}
for z in zona_unik:
    kol = np.flatnonzero(zona == z)
    seri = np.nanmean(mat_pes[:, kol], axis=1)
    indeks[z] = seri
    ambang[z] = float(np.nanpercentile(seri, P_AMBANG))

tab_ambang = pd.DataFrame({
    "zona": zona_unik,
    "bujur_min": [ringkas_zona.loc[ringkas_zona.zona == z, "lon"].min() for z in zona_unik],
    "bujur_maks": [ringkas_zona.loc[ringkas_zona.zona == z, "lon"].max() for z in zona_unik],
    "n_sel": [(zona == z).sum() for z in zona_unik],
    "swh_rata": [indeks[z].mean() for z in zona_unik],
    "ambang_m": [ambang[z] for z in zona_unik],
    "hari_terpicu_per_tahun": [(indeks[z] > ambang[z]).sum() / n_tahun for z in zona_unik],
})
tab_ambang.round(2)

## declustering

In [ ]:
def episode(x, u, jeda=JEDA):
    over = np.asarray(x) > u
    if not over.any():
        return np.array([], int), np.array([], int)
    idx = np.flatnonzero(over)
    grup = np.split(idx, np.flatnonzero(np.diff(idx) > jeda) + 1)
    return np.array([g[0] for g in grup]), np.array([len(g) for g in grup])

MUSIM = {12: "Barat", 1: "Barat", 2: "Barat", 3: "Peralihan I", 4: "Peralihan I",
         5: "Peralihan I", 6: "Timur", 7: "Timur", 8: "Timur",
         9: "Peralihan II", 10: "Peralihan II", 11: "Peralihan II"}

baris_ep = []
for z in zona_unik:
    mulai, durasi = episode(indeks[z], ambang[z])
    for m, d in zip(mulai, durasi):
        tgl = waktu[m]
        baris_ep.append({"zona": z, "tanggal_mulai": tgl, "durasi_hari": int(d),
                         "tahun": tgl.year, "bulan": tgl.month, "musim": MUSIM[tgl.month],
                         "hari_ke": int(tgl.dayofyear), "swh_puncak": float(
                             indeks[z][m:m + d].max())})
df_ep = pd.DataFrame(baris_ep).sort_values(["zona", "tanggal_mulai"]).reset_index(drop=True)

print(f"{len(df_ep)} episode di {len(zona_unik)} zona selama {n_tahun} tahun")
print(f"durasi: rata-rata {df_ep.durasi_hari.mean():.2f} hari, "
      f"maks {df_ep.durasi_hari.max()}, varians {df_ep.durasi_hari.var():.2f}")
df_ep.head(8)

### dispersion: days vs episodes

In [ ]:
bulan_kunci = pd.PeriodIndex(waktu, freq="M")
semua_bulan = pd.period_range(bulan_kunci.min(), bulan_kunci.max(), freq="M")

baris_cacah, baris_disp = [], []
for z in zona_unik:
    terpicu = indeks[z] > ambang[z]
    n_hari = pd.Series(terpicu, index=bulan_kunci).groupby(level=0).sum().reindex(
        semua_bulan, fill_value=0)
    ep_z = df_ep[df_ep.zona == z]
    n_ep = (pd.Series(1, index=pd.PeriodIndex(ep_z.tanggal_mulai, freq="M"))
            .groupby(level=0).sum().reindex(semua_bulan, fill_value=0))
    panjang = pd.Series(semua_bulan.days_in_month, index=semua_bulan)

    for b in semua_bulan:
        baris_cacah.append({"zona": z, "tahun": b.year, "bulan": b.month,
                            "n_episode": int(n_ep[b]), "n_hari_terpicu": int(n_hari[b]),
                            "hari_dalam_bulan": int(panjang[b]), "musim": MUSIM[b.month]})
    baris_disp.append({"zona": z, "ambang_m": ambang[z],
                       "disp_hari": n_hari.var() / n_hari.mean(),
                       "disp_episode": n_ep.var() / n_ep.mean()})

df_cacah = pd.DataFrame(baris_cacah)
df_disp = pd.DataFrame(baris_disp)
df_disp.round(2)

## duration

In [ ]:
d_obs = df_ep.durasi_hari.values

def pmf_geom(d, q):
    return q * (1 - q) ** (d - 1)

def pmf_ztnb(d, r, p):
    pm = stats.nbinom.pmf(d, r, p)
    return pm / (1 - stats.nbinom.pmf(0, r, p))

def pmf_dweib(d, q, b):
    return q ** ((d - 1) ** b) - q ** (d ** b)

def fit(pmf, x0, bounds):
    nll = lambda th: -np.log(np.clip(pmf(d_obs, *th), 1e-300, None)).sum()
    r = optimize.minimize(nll, x0, bounds=bounds, method="L-BFGS-B")
    return r.x, -r.fun

hasil_fit = []
for nama, pmf, x0, bnd in [
        ("Geometrik",        pmf_geom,  [0.35],       [(1e-4, 1 - 1e-4)]),
        ("NB terpotong-nol", pmf_ztnb,  [2.0, 0.45],  [(0.05, 50), (1e-3, 1 - 1e-3)]),
        ("Weibull diskret",  pmf_dweib, [0.7, 1.1],   [(1e-3, 1 - 1e-3), (0.2, 5)])]:
    th, ll = fit(pmf, x0, bnd)
    hasil_fit.append({"distribusi": nama, "parameter": np.round(th, 4).tolist(),
                      "loglik": ll, "AIC": 2 * len(th) - 2 * ll})

df_fit = pd.DataFrame(hasil_fit).sort_values("AIC").reset_index(drop=True)
mu_D = d_obs.mean()
print(f"mu_D = {mu_D:.3f} hari | E[D^2] = {(d_obs**2).mean():.3f} | "
      f"rasio var/mean = {d_obs.var()/mu_D:.3f} | extremal index theta = 1/mu_D = {1/mu_D:.3f}")
df_fit.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
maks = d_obs.max()
xs = np.arange(1, maks + 1)
ax.hist(d_obs, bins=np.arange(0.5, maks + 1.5), density=True,
        color="#c9d6e3", edgecolor="w", label="teramati")
terbaik = df_fit.iloc[0]
pmf_map = {"Geometrik": pmf_geom, "NB terpotong-nol": pmf_ztnb, "Weibull diskret": pmf_dweib}
for nama in df_fit.distribusi:
    th = df_fit.loc[df_fit.distribusi == nama, "parameter"].iloc[0]
    ax.plot(xs, pmf_map[nama](xs, *th), marker="o", ms=3, lw=1,
            label=f"{nama} (AIC {df_fit.loc[df_fit.distribusi==nama,'AIC'].iloc[0]:.1f})")
ax.set_xlabel("durasi episode (hari)"); ax.set_ylabel("peluang")
ax.set_title(f"Distribusi durasi — {len(d_obs)} episode, {n_tahun} tahun", fontsize=10)
ax.legend(fontsize=7, frameon=False); fig.tight_layout()
fig.savefig(FIGURES / f"fig_durasi_episode_wpp{WPP}.png", dpi=160, bbox_inches="tight")
plt.show()

## cyclic Poisson GLM

In [ ]:
def desain(hari_ke, J):
    X = [np.ones(len(hari_ke))]
    for j in range(1, J + 1):
        X += [np.cos(2 * np.pi * j * hari_ke / T), np.sin(2 * np.pi * j * hari_ke / T)]
    return np.column_stack(X)

tengah = pd.to_datetime(dict(year=df_cacah.tahun, month=df_cacah.bulan, day=15)).dt.dayofyear.values

baris_lr = []
lam_kurva = {}
for z in zona_unik:
    m = df_cacah.zona == z
    y = df_cacah.loc[m, "n_episode"].values
    off = np.log(df_cacah.loc[m, "hari_dalam_bulan"].values)
    hk = tengah[m.values]

    fits = {}
    for J in (0, 1, 2):
        fits[J] = sm.GLM(y, desain(hk, J), family=sm.families.Poisson(), offset=off).fit()

    lr1 = 2 * (fits[1].llf - fits[0].llf)
    lr2 = 2 * (fits[2].llf - fits[0].llf)
    J_terbaik = min((0, 1, 2), key=lambda J: fits[J].aic)
    baris_lr.append({"zona": z, "LR_J1": lr1, "p_J1": stats.chi2.sf(lr1, 2),
                     "LR_J2": lr2, "p_J2": stats.chi2.sf(lr2, 4),
                     "AIC_J0": fits[0].aic, "AIC_J1": fits[1].aic, "AIC_J2": fits[2].aic,
                     "J_terbaik": J_terbaik})

    hari = np.arange(1, 366)
    lam_kurva[z] = fits[J_terbaik].predict(desain(hari, J_terbaik), offset=np.zeros(365))

df_lr = pd.DataFrame(baris_lr)
df_lr.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.4))
warna = plt.cm.viridis(np.linspace(0, .85, len(zona_unik)))
for c, z in zip(warna, zona_unik):
    ax.plot(np.arange(1, 366), lam_kurva[z] * 30.4, color=c, lw=1.4,
            label=f"Z{z} ({tab_ambang.loc[tab_ambang.zona==z,'ambang_m'].iloc[0]:.2f} m)")
for b, nm in [(1, "Barat"), (91, "Peralihan I"), (152, "Timur"), (244, "Peralihan II")]:
    ax.axvline(b, color="k", lw=0.4, alpha=0.25)
    ax.annotate(nm, (b + 3, ax.get_ylim()[1]), fontsize=7, va="top", color="#555")
ax.set_xlabel("hari ke- dalam tahun"); ax.set_ylabel("episode per bulan")
ax.set_title("Gambar 3.3 — intensitas siklis λ(t) per zona, WPP 573", fontsize=10)
ax.legend(fontsize=7, frameon=False, ncol=2); ax.margins(x=0.01)
fig.tight_layout()
fig.savefig(FIGURES / f"fig_intensitas_siklis_wpp{WPP}.png", dpi=160, bbox_inches="tight")
plt.show()

## save

In [ ]:
baris_harian = []
for z in zona_unik:
    baris_harian.append(pd.DataFrame({
        "tanggal": waktu, "zona": z, "swh_indeks": indeks[z],
        "ambang_m": ambang[z], "terpicu": (indeks[z] > ambang[z]).astype(int),
    }))
df_harian = pd.concat(baris_harian, ignore_index=True)
df_harian["tahun"] = df_harian.tanggal.dt.year
df_harian["bulan"] = df_harian.tanggal.dt.month
df_harian["musim"] = df_harian.bulan.map(MUSIM)
df_harian = df_harian[["tanggal", "zona", "swh_indeks", "ambang_m", "terpicu",
                       "tahun", "bulan", "musim"]]

df_diag = df_disp.merge(df_lr, on="zona").merge(
    tab_ambang[["zona", "n_sel", "hari_terpicu_per_tahun"]], on="zona")
df_diag["mu_D_zona"] = [df_ep.loc[df_ep.zona == z, "durasi_hari"].mean() for z in df_diag.zona]
df_diag["theta_zona"] = 1 / df_diag["mu_D_zona"]

simpan = [
    (df_harian,   PROCESSED / f"swh_harian_zona_wpp{WPP}.csv"),
    (tab_ambang,  PROCESSED / f"ambang_zona_wpp{WPP}.csv"),
    (df_ep,       PROCESSED / f"episode_wpp{WPP}.csv"),
    (df_cacah,    PROCESSED / f"cacah_bulanan_zona_wpp{WPP}.csv"),
    (df_fit,      TABLES    / f"durasi_fit_wpp{WPP}.csv"),
    (df_diag,     TABLES    / f"diagnostik_wpp{WPP}.csv"),
]
for d, p in simpan:
    d.to_csv(p, index=False, float_format="%.5f")
    print(f"{p.relative_to(ROOT)!s:52s} {len(d):6,} baris x {d.shape[1]} kolom")

## summary

In [ ]:
print(f"Jendela kalibrasi : {TAHUN_AWAL}-{TAHUN_AKHIR} ({n_tahun} tahun, {len(waktu):,} hari)")
print(f"Zona              : {len(zona_unik)} zona bujur {LEBAR_ZONA:.0f} derajat")
print(f"Ambang p{P_AMBANG}         : {tab_ambang.ambang_m.min():.2f} - {tab_ambang.ambang_m.max():.2f} m")
print(f"Episode           : {len(df_ep)} total, rata-rata durasi {mu_D:.2f} hari")
print(f"Extremal index    : {1/mu_D:.3f}")
print(f"Dispersi hari     : {df_disp.disp_hari.min():.2f} - {df_disp.disp_hari.max():.2f}")
print(f"Dispersi episode  : {df_disp.disp_episode.min():.2f} - {df_disp.disp_episode.max():.2f}")
print(f"Siklisitas        : {(df_lr.p_J2 < 0.05).sum()}/{len(df_lr)} zona menolak Poisson homogen (p<0,05)")
print(f"Distribusi durasi : {df_fit.distribusi.iloc[0]} (AIC {df_fit.AIC.iloc[0]:.1f})")